# emission

> Graph-root emission (CR-18 revolution 2): a completed source EMITS `Source -> AudioSegment -> Transcript` into the shared context graph — the graph now BEGINS at transcription (where-graph-begins resolution: ingestion is the first EXTENDER that plants the root). Deterministic identity tuples make emission idempotent: re-runs (cache hits included) collide into verified no-ops instead of duplicating roots (the E13 hazard, relocated into graph creation and discharged).

In [ ]:
#| default_exp emission

In [ ]:
#| export
import logging
from typing import Any, Dict, List, Optional, Tuple

from cjm_plugin_system.core.queue import JobQueue
from cjm_context_graph_layer.grammar import spine_edges
from cjm_context_graph_layer.ops import extend_graph
from cjm_context_graph_layer.declare import Derivation, derivation_to_graph
from cjm_transcript_graph_schema.schema import SourceNode, AudioSegmentNode, TranscriptNode

from cjm_transcription_core.models import SourceResult

logger = logging.getLogger(__name__)

In [ ]:
#| export
def build_source_emission(
    src: SourceResult,                          # Completed per-source pipeline result (0.2.0 shape)
    transcriber_config_hashes: Dict[str, str],  # transcriber -> effective config hash (Transcript identity input)
    preprocessing: Optional[str] = None,        # Audio-preprocessing descriptor (e.g. "cjm-media-plugin-demucs@<cfg12>"); None = none applied
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], Dict[str, Any]]:  # (nodes, edges, ids)
    """Build the graph-root payload for one source (pure; no capability calls).

    Emits the locked layer schema: one `Source` (identity = file content hash)
    -> coarse `AudioSegment` spine (PART_OF / NEXT / STARTS_WITH via
    `spine_edges`) -> per-transcriber `Transcript` variants (DERIVED_FROM their
    AudioSegment; identity mirrors the capability cache key). Returns the ids
    dict {"source", "audio_segments", "transcripts"} for callers (decomp
    recomputes these same ids from the manifest — no stored-id coupling).

    `preprocessing` (when set) is recorded as a NON-IDENTITY descriptor property
    on each AudioSegment so a single-config preprocessed graph is self-describing
    (which audio-preprocessing produced its model-input). INTERIM pending the
    AudioRendition-node schema work ([[audio-rendition-node-deferred]]): the
    model_input_hash already differs raw-vs-vocals (the SourceRef content-hash),
    so raw + preprocessed AudioSegments cannot coexist in one graph today (the
    layer's verify-if-present loud-fails on the content-hash mismatch); this
    property just labels the variant within its own (separate) graph.
    """
    if not src.content_hash:
        raise ValueError(f"source {src.source_path} has no content_hash — emission identity requires it")
    source = SourceNode(content_hash=src.content_hash, path=src.source_path)
    nodes: List[Dict[str, Any]] = [source.to_graph_node()]
    edges: List[Dict[str, Any]] = []
    aseg_ids: List[str] = []
    transcript_ids: Dict[str, List[str]] = {}

    for rec in src.segments:
        if not rec.model_input_hash:
            raise ValueError(f"segment {rec.index} has no model_input_hash — emission identity requires it")
        aseg = AudioSegmentNode(
            source=source.id, index=rec.index, start=rec.start, end=rec.end,
            model_input_path=rec.model_input_path, model_input_hash=rec.model_input_hash,
            segment_path=rec.segment_path,
        )
        aseg_node = aseg.to_graph_node()
        if preprocessing:
            # Non-identity provenance label (interim; see docstring). Property adds
            # don't affect the layer's identity-mismatch check (label + sources hash).
            aseg_node["properties"]["preprocessing"] = preprocessing
        nodes.append(aseg_node)
        aseg_ids.append(aseg.id)
        for tname, tr in rec.transcripts.items():
            tnode = TranscriptNode(
                audio_segment=aseg.id, transcriber=tname,
                config_hash=transcriber_config_hashes.get(tname, ""),
                text=str(tr.get("text") or ""), audio_hash=rec.model_input_hash,
                metadata=dict(tr.get("metadata") or {}),
            )
            nodes.append(tnode.to_graph_node())
            edges.append(tnode.derived_edge())
            transcript_ids.setdefault(tname, []).append(tnode.id)

    edges = spine_edges(source.id, aseg_ids) + edges
    ids = {"source": source.id, "audio_segments": aseg_ids, "transcripts": transcript_ids}
    return nodes, edges, ids

In [ ]:
#| export
async def emit_source_graph(
    queue: JobQueue,                            # Started job queue
    graph_id: str,                              # Graph-storage capability instance id
    src: SourceResult,                          # Completed per-source pipeline result
    transcriber_config_hashes: Dict[str, str],  # transcriber -> effective config hash
    run_id: str,                                # Run id (recorded on the boundary Derivation event)
    preprocessing: Optional[str] = None,        # Audio-preprocessing descriptor (non-identity AudioSegment label); None = none
) -> Dict[str, Any]:  # Emission record for the manifest
    """Idempotently emit one source's graph root through the task channel.

    `extend_graph` = emit-if-absent + verify-if-present, so a re-run over
    cached content collides into a verified no-op (stress item 4) and a second
    transcriber's run EXTENDS the existing root (only its Transcript nodes are
    new). The host's boundary computation is declared as a `Derivation` event
    (provenance-by-declaration) ONLY when this run actually created
    AudioSegment nodes — verified re-emissions don't spam the audit trail.
    """
    nodes, edges, ids = build_source_emission(src, transcriber_config_hashes, preprocessing=preprocessing)
    res = await extend_graph(queue, graph_id, nodes, edges)
    newly_created_asegs = set(ids["audio_segments"]) & set(res.added_node_ids)
    if newly_created_asegs:
        d = Derivation(
            actor="host:cjm-transcription-core", method="segment-boundaries/v1",
            input_ids=[ids["source"]], output_ids=list(ids["audio_segments"]),
            properties={"run_id": run_id, **({"preprocessing": preprocessing} if preprocessing else {})},
        )
        dn, de = derivation_to_graph(d)
        await extend_graph(queue, graph_id, [dn], de)
    record = {
        "source_node_id": ids["source"],
        "nodes_added": res.nodes_added,
        "nodes_verified": res.nodes_verified,
        "edges_added": res.edges_added,
        "edges_existing": res.edges_existing,
    }
    logger.info(f"emitted {src.source_path}: {record}")
    return record

In [ ]:
# tests — emission payload shape + identity determinism (pure; no plugins)
from cjm_transcription_core.models import SegmentRecord
from cjm_transcript_graph_schema.schema import (
    source_node_id, audio_segment_node_id, transcript_node_id,
)

_recs = [
    SegmentRecord(index=0, start=0.0, end=280.0, duration=280.0,
                  segment_path="/cuts/s0.mp3", model_input_path="/cache/s0.wav",
                  model_input_hash="sha256:wav0",
                  transcripts={"whisper": {"job_id": "j0w", "text": "hello", "metadata": {}},
                               "voxtral": {"job_id": "j0v", "text": "hullo", "metadata": {}}}),
    SegmentRecord(index=1, start=280.0, end=560.0, duration=280.0,
                  segment_path="/cuts/s1.mp3", model_input_path="/cache/s1.wav",
                  model_input_hash="sha256:wav1",
                  transcripts={"whisper": {"job_id": "j1w", "text": "world", "metadata": {}},
                               "voxtral": {"job_id": "j1v", "text": "wurld", "metadata": {}}}),
]
_src = SourceResult(source_path="/media/ep1.mp3", duration=560.0, vad_chunk_count=99,
                    batch_key="bk", content_hash="sha256:src", segments=_recs)
_hashes = {"whisper": "sha256:cfgw", "voxtral": "sha256:cfgv"}

nodes, edges, ids = build_source_emission(_src, _hashes)
# 1 Source + 2 AudioSegment + 2x2 Transcript
assert len(nodes) == 7
labels = [n["label"] for n in nodes]
assert labels.count("Source") == 1 and labels.count("AudioSegment") == 2 and labels.count("Transcript") == 4
# spine: 1 STARTS_WITH + 2 PART_OF + 1 NEXT; plus 4 DERIVED_FROM
rels = [e["relation_type"] for e in edges]
assert rels.count("STARTS_WITH") == 1 and rels.count("PART_OF") == 2
assert rels.count("NEXT") == 1 and rels.count("DERIVED_FROM") == 4
# no preprocessing -> no descriptor property on AudioSegments
assert all("preprocessing" not in n["properties"] for n in nodes if n["label"] == "AudioSegment")

# deterministic ids recomputable from the manifest fields alone
assert ids["source"] == source_node_id("sha256:src")
a0 = audio_segment_node_id(ids["source"], 0.0, 280.0)
assert ids["audio_segments"][0] == a0
assert ids["transcripts"]["whisper"][0] == transcript_node_id(a0, "whisper", "sha256:cfgw")
# re-build -> byte-identical id sets (emission idempotency precondition)
nodes2, edges2, ids2 = build_source_emission(_src, _hashes)
assert [n["id"] for n in nodes2] == [n["id"] for n in nodes]
assert [e["id"] for e in edges2] == [e["id"] for e in edges]

# preprocessing descriptor: a NON-identity AudioSegment property — same node ids,
# property present on every AudioSegment (the interim self-describing label).
nodes_p, edges_p, ids_p = build_source_emission(_src, _hashes, preprocessing="cjm-media-plugin-demucs@cfg123")
assert ids_p["audio_segments"] == ids["audio_segments"]  # identity UNCHANGED by the property
assert [n["id"] for n in nodes_p] == [n["id"] for n in nodes]
assert all(n["properties"].get("preprocessing") == "cjm-media-plugin-demucs@cfg123"
           for n in nodes_p if n["label"] == "AudioSegment")

# identity guards fire loudly
import dataclasses
try:
    build_source_emission(dataclasses.replace(_src, content_hash=""), _hashes)
    raise AssertionError("expected ValueError")
except ValueError:
    pass
print("emission shape tests OK")